In [3]:
import os
from dotenv import load_dotenv
from google.cloud import bigquery
import pandas as pd
import numpy as np
import plotly.express as px
from  plotly.subplots import make_subplots
import plotly.graph_objects as go


In [ ]:
load_dotenv()

key_file_path = os.environ.get("GCP_KEY_PATH")

client = bigquery.Client.from_service_account_json(
    key_file_path,
    project="quantum-echo-data-eng-prod"
)

query_sales = """
    SELECT 
        product_key, 
        order_date, 
        gross_sales_amount, 
        unit_price 
    FROM `quantum-echo-data-eng-prod.gold.fct_sales`
"""

df_sales = client.query(query_sales).to_dataframe()

query_products = """
    SELECT 
        product_key,
        product_name, 
        category 
    FROM `quantum-echo-data-eng-prod.gold.dim_products`
"""

df_products = client.query(query_products).to_dataframe()


df_sales["order_date"] = pd.to_datetime(df_sales["order_date"])

# 🌟 FIX: Force raw BigQuery decimal objects into standard floats immediately
df_sales["gross_sales_amount"] = pd.to_numeric(df_sales["gross_sales_amount"], errors='coerce')
df_sales["unit_price"] = pd.to_numeric(df_sales["unit_price"], errors='coerce')

# =====================================================================
# 1. THE JOIN: Combine sales with products
# =====================================================================

df_sales_enriched = pd.merge(
    df_sales,
    df_products,
    on="product_key",
    how="left"

)
# =====================================================================
# CTE 1: Category Grain 
# =====================================================================
sales_by_category = (
    df_sales_enriched
    .assign(order_year=lambda x: pd.to_datetime(x["order_date"]).dt.year.astype(int))
    .query("order_year >= 2010") 
    .groupby(["order_year", "category"], as_index=False)
    .agg(
        category_sales=("gross_sales_amount", lambda s: pd.to_numeric(s).sum()),
        avg_price=("unit_price", lambda s: pd.to_numeric(s).mean())
    )
    .assign(
        category_sales=lambda x: x["category_sales"].round().astype("Int64"),
        avg_price=lambda x: x["avg_price"].round().astype("Int64"),
        order_year=lambda x: x["order_year"].astype(str)
    )
)

# =====================================================================
# CTE 2: Yearly Grain
# =====================================================================

yearly_sales = (
    sales_by_category
    .groupby("order_year", as_index=False)
    .agg(
        total_sales=("category_sales", "sum"),
        avg_price=("avg_price", "mean"),
    )
    .assign(
        running_total=lambda x: (x["total_sales"].cumsum()),
        moving_avg_price=lambda x: (x["avg_price"].expanding().mean())
    )
    .assign(
        total_sales=lambda x: x["total_sales"].astype("Int64"),
        avg_price=lambda x: x["avg_price"].round().astype("Int64"),
        running_total=lambda x: x["running_total"].astype("Int64"),
        moving_avg_price=lambda x: x["moving_avg_price"].round().astype("Int64")
    )
)

# =====================================================================
# CTE 3: Top N Products Grain (For Horizontal Ranking Chart)
# =====================================================================
TOP_N = 10

top_products = (
    df_sales_enriched
    .groupby("product_name", as_index=False) 
    .agg(
        product_revenue=("gross_sales_amount", lambda s: pd.to_numeric(s).sum())
    )
    .assign(
        product_revenue=lambda x: x["product_revenue"].round().astype("Int64")
    )
    # Sort from highest sales to lowest sales
    .sort_values("product_revenue", ascending=False)
    # Grab only the top N rows
    .head(TOP_N)
)

In [5]:
# --- 1. CALCULATE THE MASTER GRAND TOTAL ---
grand_total_revenue = yearly_sales["total_sales"].sum()
formatted_total = f"${grand_total_revenue:,.0f}"

# --- 2. CREATE THE SUBPLOT GRID WITH SECONDARY Y AXIS ---
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Annual Revenue & Running Total", "All-Time Category Share"),
    # 🌟 CRITICAL: Tell Column 1 it has a secondary Y-axis, Column 2 stays a donut domain
    specs=[[{"secondary_y": True}, {"type": "domain"}]],
    # 🌟 THIS FIXES THE LAYOUT: Allocate 65% width to Col 1, and 35% width to Col 2
    column_widths=[0.65, 0.35] 
)

# --- 3. COLUMN 1: ADD THE BAR CHART TRACE (Primary Y-Axis / Left) ---
fig.add_trace(
    go.Bar(
        x=yearly_sales["order_year"],
        y=yearly_sales["total_sales"],
        name="Annual Revenue Spikes",
        marker_color="#e0e0e0", # Light grey so it sits cleanly in the background
        hovertemplate="<b>Year:</b> %{x}<br><b>Annual Revenue:</b> $%{y:,.0f}<extra></extra>"
    ),
    row=1, col=1,
    secondary_y=False # 👈 Binds to the LEFT axis
)

# --- 4. COLUMN 1: ADD THE LINE CHART TRACE (Secondary Y-Axis / Right) ---
fig.add_trace(
    go.Scatter(
        x=yearly_sales["order_year"],
        y=yearly_sales["running_total"],
        name="Running Revenue Growth",
        mode="lines+markers",
        line=dict(color="crimson", width=3, shape="spline"),
        marker=dict(size=8, color="crimson", line=dict(width=2, color="white")),
        hovertemplate="<b>Year:</b> %{x}<br><b>Running Total:</b> $%{y:,.0f}<extra></extra>"
    ),
    row=1, col=1,
    secondary_y=True # 👈 Binds to the RIGHT axis
)

# --- 5. COLUMN 2: ADD THE DONUT CHART TRACE ---
# We extract the underlying raw dictionary data from px.pie to easily add it as a trace
fig_pie_base = px.pie(
    sales_by_category, 
    names="category", 
    values="category_sales",
    color_discrete_sequence=px.colors.sequential.Sunset
)

for trace in fig_pie_base.data:
    trace.update(
        hole=0.4,
        textinfo="percent", 
        textposition="inside", 
        insidetextorientation="radial",
        automargin=True,  
        hovertemplate="<b>Category:</b> %{label}<br><b>Revenue:</b> $%{value:,}<br><b>Share:</b> %{percent}<extra></extra>"
    )
    fig.add_trace(trace, row=1, col=2)

# --- 6. ADD THE HOLE ANNOTATION ---
fig.add_annotation(
    text=f"<b>Grand Total</b><br>{formatted_total}",
    x=0.850, y=0.5, showarrow=False,
    font=dict(size=11, color="#2c3e50"),
    align="center", xref="paper", yref="paper"
)

# --- 7. AXIS & CANVAS LAYOUT CONFIGURATION ---
fig.update_layout(
    title_text="Executive Performance Dashboard",
    title_x=0.5,
    template="plotly_white",
    width=1000, height=450,
    showlegend=True,
    # Merges all chart legends together cleanly at the bottom
    legend=dict(
        title=dict(text="", font=dict(size=10)), # 👈 Shrunk legend title font
        font=dict(size=10),                                    # 👈 Shrunk legend item font
       x=0.5, y=-0.20, 
        xanchor="center", yanchor="top",
        orientation="h",
        
        
    )
)

# Label your independent left and right axis rules 🏷️
fig.update_yaxes(title_text="Annual Revenue ($)", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="Running Revenue Total ($)", row=1, col=1, secondary_y=True)
fig.update_xaxes(title_text="Year", row=1, col=1)

fig.write_image("Executive_Performance_Dashboard.png", scale=2)
fig.show()


In [7]:
# 1. TOP_N variable
TOP_N = 10  

# 2. Build the standalone horizontal bar chart
fig_top_products = px.bar(
    top_products,
    x="product_revenue",
    y="product_name",
    orientation="h",
    color="product_revenue",
    color_continuous_scale=px.colors.sequential.Sunset,
    title=f"Top {TOP_N} Best Selling Products by All-Time Revenue"

)

# 3. Clean up the hover tooltips and trace metrics
fig_top_products.update_traces(
    hovertemplate="<b>Product:</b> %{y}<br><b>Revenue:</b> $%{x:,.0f}<extra></extra>"
)

# 4. Polish the canvas layout grid properties
fig_top_products.update_layout(
    template="plotly_white",
    width=800,
    height=500,
    coloraxis_showscale=False,       # Hides the secondary colorbar scale to keep it clean
    title_x=0.5,                      # Centers the main title text
    margin=dict(l=150)                # Adds left margin space so long product names aren't cut off
)

# 5. Format axes and force sorting sequence
fig_top_products.update_xaxes(title_text="Total Revenue ($)")
fig_top_products.update_yaxes(
    title_text="", 
    categoryorder="total ascending"   # 👈 Forces the highest earner to stay at the very top
)
# 6. Save and Display
fig_top_products.write_image("Best_Selling_Products.png", scale=2)
fig_top_products.show()